# Credential-Sharing Graph — EDA

Exploration only. Any logic worth keeping graduates into `graph.py` / `charts.py` as a named function — this notebook should only ever call into those modules, not reimplement analysis logic inline.

**Setup** (see `../README.md` for the full version): run once per machine —
```
uv sync --group test --group analysis
uv run python -m ipykernel install --user --name bear-trap-analysis --display-name "Python (bear-trap analysis)" --env PYTHONPATH "$(pwd)"
```
then select the "Python (bear-trap analysis)" kernel for this notebook — the same one `fingerprinting/notebooks/eda.ipynb` uses. Both notebooks import via absolute `honeypot.analysis.*` package paths, so one shared kernel (repo root on `PYTHONPATH`) works for both; there's no per-submodule scoping needed anymore.

See `.claude/tasks/active-task.md` for the full spec (local/gitignored — won't exist on a fresh clone).

In [ ]:
import polars as pl

from honeypot.analysis.common.extract import get_sessions
from honeypot.analysis.network.charts import (
    plot_cluster_size_distribution,
    plot_top_clusters_subgraph,
)
from honeypot.analysis.network.graph import (
    assign_clusters,
    build_credential_graph,
    build_ip_similarity_graph,
)

## Load data

`"offline"` reads the most recent local snapshot from the shared `../common/outputs/snapshots/` (no GCP credentials needed). Switch to `"live"` to pull fresh from BigQuery and refresh the snapshot as a side effect.

In [ ]:
MODE = "offline"

events = get_sessions(MODE)
events.select("src_ip", "username", "password").head()

## Build the credential-sharing graph

Two filtering stages, see `../README.md` for the full "why" with real
numbers:
1. `build_credential_graph` excludes credential pairs attempted by
   `>= min_ips_to_exclude` distinct IPs (default 20) -- without this, a
   pair like `admin`/`admin` acts as a hub node merging every IP that ever
   tried it.
2. `build_ip_similarity_graph` projects the bipartite graph onto IPs
   alone, linking two IPs only if their TF-IDF cosine similarity is
   `>= min_similarity` (default 0.1) **and** they share `>=
   min_shared_pairs` credential pairs (default 2). Each check alone lets a
   different kind of weak link chain IPs into a giant component even after
   (1): cosine alone links on one coincidental rare pair, and the count
   alone links big dictionaries that barely overlap.

`assign_clusters` runs `networkx.connected_components` over the
*similarity* graph (not the raw bipartite one) to group related IPs.

In [ ]:
graph = build_credential_graph(events)
similarity_graph = build_ip_similarity_graph(graph)
clusters = assign_clusters(similarity_graph)

print(f"bipartite: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
print(
    f"similarity: {similarity_graph.number_of_nodes()} nodes, "
    f"{similarity_graph.number_of_edges()} edges"
)
print(f"{clusters.height} IPs clustered")
print(f"{clusters['cluster_id'].n_unique()} distinct clusters")
clusters.sort("cluster_size", descending=True).head(10)

In [ ]:
cluster_zero = clusters.filter(pl.col("cluster_id") == 0)
events_cluster_zero = events.filter(
    pl.col("src_ip").is_in(cluster_zero["src_ip"].to_list())
)
events_cluster_zero.select("src_ip", "username", "password").unique()

In [ ]:
events.filter(pl.col("username") == "grok")

## Baseline charts

`charts.py` forces the `Agg` backend (headless-safe for tests/`pipeline.py`), so these calls save a PNG but won't render inline on their own — wrap with `Image(...)` to preview.

In [ ]:
from IPython.display import Image

Image(plot_cluster_size_distribution(clusters))

In [ ]:
path = plot_top_clusters_subgraph(similarity_graph)
if path:
    display(Image(path))
else:
    print("Skipped -- see the message above: subgraph too large to plot.")

## Exploration

Ad hoc from here — how much a handful of generic credential pairs dominate the largest clusters (the risk flagged in the module's spec), whether cluster membership correlates with `cloud_provider`/`sensor` (same campaign hitting both clouds), whatever the data suggests.

In [ ]:
clusters.filter(pl.col("cluster_size") == 1).height

## What got excluded: high-frequency credential pairs

Pairs excluded by `min_ips_to_exclude` aren't just dropped silently --
`generic_credential_attempts` surfaces them separately, by how many
distinct IPs attempted each one. Useful for sanity-checking the threshold
itself (does it look like real weak-credential/protocol-noise traffic, or
is something legitimate getting excluded?).

In [ ]:
from honeypot.analysis.network.charts import plot_generic_credential_breakdown
from honeypot.analysis.network.graph import generic_credential_attempts

generic_attempts = generic_credential_attempts(events)
Image(plot_generic_credential_breakdown(generic_attempts))

## Who's targeting which app: username breakdown

Independent of the graph entirely -- `username_targeting_breakdown` just
counts distinct IPs per username. This is the answer to "who's looking for
specific apps" (`postgres`, `mysql`, `grok`, ...): a widely-known service
gets probed with the same handful of obvious passwords by many unrelated
actors, so at the *pair* level it's exactly as noisy as `root`/`admin` --
clustering can never preserve this signal, no matter how the threshold is
tuned. Username-level counting sidesteps the problem instead of fighting
it.

In [ ]:
from honeypot.analysis.network.charts import plot_username_targeting_breakdown
from honeypot.analysis.network.graph import username_targeting_breakdown

username_breakdown = username_targeting_breakdown(events)
Image(plot_username_targeting_breakdown(username_breakdown))

## Zooming in: AI-tooling usernames

`ai_targeting.py` narrows the username breakdown to accounts tied to AI
tooling -- coding agents (`claude`, `codex`, `openclaw`, ...), model
vendors (`grok`, `gemini`), LLM runtimes (`ollama`, `vllm`) and GPU/ML
hosts -- matched on whole username tokens via `AI_USERNAME_KEYWORDS`, not
raw substrings. Bars are colored by category.

In [ ]:
from honeypot.analysis.network.ai_targeting import (
    ai_targeting_ips,
    ai_username_targeting,
)
from honeypot.analysis.network.charts import plot_ai_username_targeting

ai_usernames = ai_username_targeting(events)
Image(plot_ai_username_targeting(ai_usernames))

### Dedicated AI hunters vs. broad dictionaries

`ai_share` = AI usernames / all usernames an IP tried. Low (~0.04) means a
big generic dictionary that happens to include AI names; high means an IP
actually hunting AI tooling -- those are the leads worth reading by hand.
A null `cluster_id` means every pair the IP tried was excluded from the
graph as generic.

In [ ]:
ai_ips = ai_targeting_ips(events, clusters=clusters)

print(f"{ai_ips.height} IPs tried at least one AI username")
ai_ips.filter(pl.col("ai_share") >= 0.25).sort("ai_share", descending=True)